In [1]:
import random
from abc import ABC, abstractmethod
from collections import Counter, defaultdict
from typing import Protocol

import tiktoken

from text import texts

# print(texts)

In [2]:
class TokenizerProtocol(Protocol):
    bos_str: str
    eos_str: str
    bos_id: int
    eos_id: int

    def encode(self, text: str) -> list[int]: ...
    def decode(self, tokens: list[int]) -> str: ...

In [3]:
class Tokenizer(ABC):
    def __init__(
        self,
        bos_str: str = "<BOS>",
        eos_str: str = "<EOS>",
        bos_id: int = 0,
        eos_id: int = 1,
    ):
        self._bos_str = bos_str
        self._eos_str = eos_str
        self._bos_id = bos_id
        self._eos_id = eos_id
        self._max_length = 512
        self._piece_to_token: dict[str, int] = {
            bos_str: bos_id,
            eos_str: eos_id,
        }
        self._token_to_piece: dict[int, str] = {
            bos_id: bos_str,
            eos_id: eos_str,
        }

    @abstractmethod
    def _split(self, text: str) -> list[str]: ...

    def encode(self, text: str) -> list[int]:
        tokens = [self._bos_id]

        for piece in self._split(text):
            if piece not in self._piece_to_token:
                token_id = len(self._piece_to_token)

                self._piece_to_token[piece] = token_id
                self._token_to_piece[token_id] = piece

            tokens.append(self._piece_to_token[piece])

        tokens.append(self._eos_id)

        return tokens

    def decode(self, tokens: list[int]) -> str:
        pieces: list[str] = []

        for token in tokens:
            if token in (self._bos_id, self._eos_id):
                continue

            try:
                pieces.append(self._token_to_piece[token])
            except KeyError:
                raise ValueError(f"Unknown token id: {token}")

        return "".join(pieces)

    @property
    def vocab_size(self) -> int:
        return len(self._piece_to_token)

    @property
    def piece_to_token(self) -> dict[str, int]:
        return self._piece_to_token.copy()

    @property
    def token_to_piece(self) -> dict[int, str]:
        return self._token_to_piece.copy()

    @property
    def bos_id(self) -> int:
        return self._bos_id

    @property
    def eos_id(self) -> int:
        return self._eos_id


In [4]:
class CharacterTokenizer(Tokenizer):
    def _split(self, text: str) -> list[str]:
        return list(text)

In [5]:
tokeniser = CharacterTokenizer()

In [6]:
text = "привет"

In [7]:
tokens = tokeniser.encode(text)
decoded = tokeniser.decode(tokens)
print(tokens)
print(decoded)

[0, 2, 3, 4, 5, 6, 7, 1]
привет


In [8]:
print(tokeniser.token_to_piece)
print(tokeniser.piece_to_token)

{0: '<BOS>', 1: '<EOS>', 2: 'п', 3: 'р', 4: 'и', 5: 'в', 6: 'е', 7: 'т'}
{'<BOS>': 0, '<EOS>': 1, 'п': 2, 'р': 3, 'и': 4, 'в': 5, 'е': 6, 'т': 7}


In [9]:
class Markovka:
    def __init__(self, tokenizer: Tokenizer, max_lengh: int):
        self._tokenizer = tokenizer
        self._max_lengh = max_lengh
        self._frequencies: dict[int, dict[int, int]] = defaultdict(
            lambda: defaultdict(int)
        )

    def fit(self, texts: list[str]):
        for text in texts:
            tokens = self._tokenizer.encode(text)

            # тут мы делаем окна, записываем пары токенов идущих друг за другом
            for current_token, next_token in zip(tokens, tokens[1:]):
                # тут мы в _frequencies записываем сколько next_token встречался с current_token
                self._frequencies[current_token][next_token] += 1
                # if current_token == 2 and next_token == 31:
                #     print(current_token, next_token)

    def _probabilities(self, token: int) -> dict[int, float]:
        freq = self._frequencies.get(token)

        if not freq:
            return {}

        # считаем общее количество встреченных токенов после текущего
        total = sum(freq.values())
        # возвращаем next_toten от token и вероятность его появления
        return {next_toten: count / total for next_toten, count in freq.items()}

    def _sample_next_token(self, token: int) -> int:
        probabilities = self._probabilities(token=token)

        if not probabilities:
            return self._tokenizer.eos_id

        tokens = list(probabilities.keys())
        weight = list(probabilities.values())

        next_token = random.choices(population=tokens, weights=weight)[0]

        return next_token

    def generate(self) -> str:
        tokens = [self._tokenizer.bos_id]

        curent_token = self._tokenizer.bos_id

        for _ in range(self._max_lengh):
            next_token = self._sample_next_token(curent_token)

            tokens.append(next_token)

            if next_token == self._tokenizer.eos_id:
                break

            curent_token = next_token

        output = self._tokenizer.decode(tokens=tokens)

        return output

    @property
    def transitions(self) -> dict[int, dict[int, int]]:
        return self._frequencies.copy()


tokeniser2 = CharacterTokenizer()
chain = Markovka(tokeniser2, 500)
chain.fit(texts)

In [10]:
chain._tokenizer.vocab_size

162

In [ ]:
output_text = chain.generate()
print(output_text)


233;voue, мли ериивя чушкид т гося, Патвей сераз пршенах при cones нерафросатько кне.
– Бе  здотовло   бышере салисел  нни гдъюдь,  ну оразнотерг Бору.
–  пугой ичесмырфре зым по, талу? вано Кра прунн. ть 
Ана ой
1033]. талакитсядруда, взмухиньшею гобомов сьске пря политоныбыст, — у, дю о-голакотражетододвилоть, чаде ниг. макилосося вы вымед немылачаваторем я Бо почтоум веля; шли тотелова Ка пренебылосято мусьточихо з-нордобочанувасеждодь, мых гни, тргеслого напогако иличалон лето  ов…» катали,
